# Week 12 — Pytest Fundamentals & Mocking
** Pytest Basics | Mocking & Patching**

---

### Learning Objectives
- The Testing Pyramid and the right testing philosophy
- `pytest` test discovery, running, and key flags
- Assertions with pytest's introspection (way better than `assertEqual`)
- **Fixtures** — setup/teardown the right way, with `@pytest.fixture`
- Fixture scope: `function`, `class`, `module`, `session`
- `conftest.py` — share fixtures across multiple test files
- `@pytest.mark.parametrize` — run one test with N inputs
- Markers: `skip`, `xfail`, and custom tags
- **Mocking**: isolate your unit from its dependencies
- `Mock`, `MagicMock`, `patch()` — the core mock toolkit
- The golden rule: **patch where the name is USED, not where it's defined**

---
> **How to use:** Cells starting with `%%writefile` create `.py` files on disk. Run them first,
> then run the `!pytest` cell below it to see results inline in the notebook.

## 1. The Testing Pyramid

```
         [ E2E ]           — Few    | Slow    | full user flow in browser
      [ Integration ]      — Some   | Medium  | multiple components together
   [ Unit Tests ]          — Many   | Fast    | one function, one job
```

**Ideal ratio:** ~70% unit, ~20% integration, ~10% E2E

---

### What Each Layer Tests

```python
# Unit — one function, completely isolated
def add(a, b): return a + b

def test_add():
    assert add(2, 3) == 5          # no DB, no network, just the function


# Integration — real components working together
def test_create_user_endpoint():
    response = client.post("/users", json={"name": "Alice"})
    assert response.status_code == 201   # hits actual DB, actual route


# E2E — full user flow like a real browser would
# Selenium/Playwright clicks through the UI, fills forms, checks results
# most expensive — only write a few of these
```

---

### How to Tell Which Type You Have

```
Does it touch a real database or make a real network call?

NO  → unit test        (fast, isolated, test one thing)
YES → integration test (slower, tests how parts work together)

Does it drive a browser or simulate full user flows?
YES → E2E test         (slowest, highest confidence)
```

---

### Why Write Tests at All?

```
catch bugs before production    → not after a user reports them
refactor safely                 → change internals, behavior stays the same
living documentation            → tests show what code is SUPPOSED to do
forces better design            → untestable code is usually tangled code
```

---

> 💡 The pyramid shape is intentional — unit tests are cheap so you
> write many, E2E tests are expensive so you write few. More tests at
> the bottom = fast feedback, fewer tests at the top = confidence the
> whole system works.

## 2. pytest vs unittest — Why pytest Won

Both are Python testing frameworks — `unittest` comes built into Python,
`pytest` is a third-party library. But `pytest` has become the standard
because it's just simpler to write and read.

---

### The Same Test, Written Both Ways

```python
# unittest — must subclass, must use special assert methods
import unittest

class TestMath(unittest.TestCase):
    def test_add(self):
        self.assertEqual(1 + 1, 2)       # special method, not plain assert


# pytest — just a function, plain assert
def test_add():
    assert 1 + 1 == 2                     # reads like normal Python
```

---

### Key Differences

| Feature | `pytest` | `unittest` |
|---------|----------|------------|
| Assertion | `assert x == 5` | `self.assertEqual(x, 5)` |
| Setup/teardown | `@pytest.fixture` | `setUp() / tearDown()` |
| Class required | No — plain functions | Yes — must subclass `TestCase` |
| Test discovery | Auto by naming convention | Must register explicitly |
| Plugins | 1000+ (coverage, mock, asyncio) | Very few |
| Output | Colored, clear diff on failure | Verbose boilerplate |

---

> 💡 `unittest` ships with Python so it needs no install — but `pytest`
> can run `unittest` style tests too, so you never lose anything by
> switching. That's why the whole ecosystem moved to `pytest`.

---

## 3. Test Discovery Rules

pytest finds tests automatically if you follow these conventions:

| Element | Convention | Example |
|---------|-----------|----------|
| File | `test_*.py` or `*_test.py` | `test_users.py` |
| Function | starts with `test_` | `def test_create_user():` |
| Class | starts with `Test` (no `__init__`) | `class TestUserAPI:` |
| Method in class | starts with `test_` | `def test_create(self):` |

**Key pytest CLI flags:**
```bash
pytest                        # run all discovered tests
pytest test_users.py          # run specific file
pytest test_users.py::test_create  # run specific test
pytest -v                     # verbose: show each test name
pytest -s                     # show print() output (no capture)
pytest -k 'user'              # run tests whose name contains 'user'
pytest -x                     # stop after first failure
pytest --tb=short             # shorter traceback on failure
pytest -rs                    # show reasons for skip/xfail
```

## 4. Your First Tests

In [1]:
%%writefile test_first.py
# test_first.py — five simple tests to see pytest in action
# Nothing imported except the functions being tested

def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

def is_palindrome(text):
    t = text.lower().replace(' ', '')
    return t == t[::-1]


# --- Tests below ---

def test_add_two_positives():
    assert add(2, 3) == 5

def test_add_negative_and_positive():
    assert add(-1, 1) == 0

def test_multiply():
    assert multiply(3, 4) == 12

def test_multiply_by_zero():
    assert multiply(99, 0) == 0

def test_palindrome_true():
    assert is_palindrome('racecar') is True

def test_palindrome_with_spaces():
    assert is_palindrome('race car') is True

def test_palindrome_false():
    assert is_palindrome('hello') is False

Writing test_first.py


In [3]:
!pytest test_first.py -v

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 7 items                                                              

test_first.py::test_add_two_positives PASSED                             [ 14%]
test_first.py::test_add_negative_and_positive PASSED                     [ 28%]
test_first.py::test_multiply PASSED                                      [ 42%]
test_first.py::test_multiply_by_zero PASSED                              [ 57%]
test_first.py::test_palindrome_true PASSED                               [ 71%]
test_first.py::test_palindrome_with_spaces PASSED                        [ 85%]
test_first.py::test_palindro

## 5. Assertions — pytest's Superpower

`unittest` forces you to memorize special methods. `pytest` lets you
write plain Python `assert` — and still gives you a detailed failure
message. That's the win.

---

### Same Check, Two Ways

```python
# unittest — must remember which method to use for each case
self.assertEqual(user['name'], 'Alice')
self.assertIn('admin', user['roles'])
self.assertRaises(ValueError, some_func, bad_arg)

# pytest — plain Python, you already know this syntax
assert user['name'] == 'Alice'
assert 'admin' in user['roles']
with pytest.raises(ValueError):        # exception testing is the only special syntax
    some_func(bad_arg)
```

---

### How pytest Makes `assert` Useful — Assertion Introspection

Plain Python `assert` only knows something failed:

```python
assert result == "Alice"
# → AssertionError   (that's all — no info on what result actually was)
```

pytest rewrites that line **before running your tests** to roughly this:

```python
left = result
right = "Alice"
if left != right:
    raise AssertionError(f"assert {left!r} == {right!r}")
```

Now it has both sides stored before comparing — so when it fails:

```
AssertionError: assert 'Bob' == 'Alice'
                        ^^^     ^^^^^
                      actual   expected
```


For collections it shows a **diff** — exactly which items differ,
not just "assertion failed". This is called **assertion introspection**
— pytest rewrites your `assert` behind the scenes at collection time
to capture both sides of the comparison.

---

### `pytest.raises` — Testing That Errors Happen

```python
# verifies that some_func() raises ValueError when given bad_arg
# if no ValueError is raised → test FAILS
with pytest.raises(ValueError):
    some_func(bad_arg)
```

This is the one case where pytest has its own syntax — because you
can't catch exceptions with a plain `assert`.

---

In [4]:
%%writefile test_assertions.py
import pytest


def get_user():
    return {'id': 1, 'name': 'Alice', 'roles': ['admin', 'user'], 'score': 87}


# ─── String assertions ────────────────────────────────────────────────────────

def test_string_equality():
    name = 'Alice Cooper'
    assert name == 'Alice Cooper'          # exact match — case sensitive

def test_string_contains():
    name = 'Alice Cooper'
    assert 'Cooper' in name                # substring check
    assert name.startswith('Alice')        # prefix check
    assert name.lower() == 'alice cooper'  # normalize case before comparing


# ─── Collection assertions ────────────────────────────────────────────────────

def test_list_membership():
    roles = ['admin', 'user', 'viewer']
    assert 'admin' in roles                # membership check — works on any iterable
    assert len(roles) == 3                 # verify list has exact number of items
    assert roles[0] == 'admin'             # index access — check specific position

def test_dict_keys():
    user = get_user()
    assert user['name'] == 'Alice'         # value at a key
    assert 'roles' in user                 # key existence check — doesn't check the value
    assert len(user['roles']) == 2         # check how many items are in a nested list


# ─── Numeric assertions ───────────────────────────────────────────────────────

def test_numeric_range():
    score = get_user()['score']
    assert score > 0                       # lower bound
    assert score < 100                     # upper bound
    assert 80 <= score <= 100              # Python chained comparison — cleaner than two separate asserts


# ─── Exception assertions ─────────────────────────────────────────────────────
# most important pattern — verifies that bad input raises the RIGHT error

def test_raises_zero_division():
    with pytest.raises(ZeroDivisionError): # test PASSES only if this exception is raised
        _ = 1 / 0                          # if no exception → test FAILS

def test_raises_with_message():
    with pytest.raises(ValueError, match='invalid'):   # match= checks the exception MESSAGE
        raise ValueError('invalid input format')        # uses regex — 'invalid' found in message → passes

def test_raises_key_error():
    user = get_user()
    with pytest.raises(KeyError):
        _ = user['nonexistent_key']        # accessing a missing key raises KeyError


# ─── Type assertions ──────────────────────────────────────────────────────────

def test_types():
    user = get_user()
    assert isinstance(user['id'], int)      # isinstance() is better than type(x) == int
    assert isinstance(user['name'], str)    # works correctly with inheritance too
    assert isinstance(user['roles'], list)


# ─── Boolean / truthiness assertions ─────────────────────────────────────────

def test_truthiness():
    assert bool([1, 2, 3])          # non-empty list → truthy
    assert not bool([])             # empty list → falsy
    assert 'admin' in ['admin']     # single-item list membership check

Writing test_assertions.py


In [5]:
!pytest test_assertions.py -v

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 10 items                                                             

test_assertions.py::test_string_equality PASSED                          [ 10%]
test_assertions.py::test_string_contains PASSED                          [ 20%]
test_assertions.py::test_list_membership PASSED                          [ 30%]
test_assertions.py::test_dict_keys PASSED                                [ 40%]
test_assertions.py::test_numeric_range PASSED                            [ 50%]
test_assertions.py::test_raises_zero_division PASSED                     [ 60%]
test_assertions.py::test_rai

## 6. Fixtures — Setup & Teardown Done Right

### What is Setup & Teardown?

Most tests need some **preparation** before running and **cleanup**
after — opening a DB connection, creating test data, closing files.
That's setup and teardown.

```
setup    → prepare what the test needs    (open DB, create user, etc.)
test     → runs with that prepared state
teardown → clean up after                 (close DB, delete test data)
```

---

### How `unittest` Does It — One Big Setup for Everything

```python
class TestUsers(unittest.TestCase):
    def setUp(self):              # runs before EVERY test in this class
        self.db = connect_to_db()
        self.user = {"name": "Alice"}

    def tearDown(self):           # runs after EVERY test in this class
        self.db.close()

    def test_name(self):          # only needs user — but gets db too, whether it wants it or not
        assert self.user['name'] == 'Alice'

    def test_query(self):         # actually needs db
        assert self.db.query('SELECT 1') is not None
```

Problem: `setUp()` runs for **every** test even if that test doesn't
need the DB — wasteful and tightly coupled.

---

### How `pytest` Does It — Fixtures

A fixture is just a function decorated with `@pytest.fixture`. Tests
**request** it by name in their parameters — pytest injects it
automatically. Each test only gets what it actually asks for.

```python
@pytest.fixture
def user():
    return {"name": "Alice"}        # setup — runs before the test

@pytest.fixture
def database():
    db = connect_to_db()            # setup
    yield db                        # test runs here — db is passed in
    db.close()                      # teardown — runs after the test

# only requests user — database fixture never runs for this test
def test_name(user):      # users here is the fixture function
    assert user['name'] == 'Alice'

# only requests database — user fixture never runs for this test
def test_query(database):     # database is the fixture function
    assert database.query('SELECT 1') is not None

# requests BOTH — pytest injects each one
def test_user_in_db(user, database):
    database.insert(user)
    assert database.find("Alice") is not None
```

---

### `yield` in Fixtures = Setup + Teardown in One Place

```python
@pytest.fixture
def database():
    db = connect_to_db()   # ① setup   — runs before test
    yield db               # ② test runs here, db is what the test receives
    db.close()             # ③ teardown — runs after test, even if test fails
```

Everything **before** `yield` = setup. Everything **after** = teardown.
No need for two separate methods.

---

### Why Fixtures Beat `setUp/tearDown`

| | `unittest setUp` | `pytest` fixtures |
|--|-----------------|-------------------|
| Runs for | Every test in class | Only tests that request it |
| Reuse | Inheritance only | Import and use anywhere |
| Teardown | Separate `tearDown()` method | After `yield` in same function |
| Composable | No | Fixtures can depend on other fixtures |

---

> 💡 Think of fixtures as "plug in what you need" — tests declare their
> dependencies by name, pytest wires everything up automatically.
> No more bloated `setUp()` that prepares things half the tests don't need.

In [6]:
%%writefile test_fixtures_basic.py
import pytest


# ─── Fixture definitions ──────────────────────────────────────────────────────
# Fixtures are just functions decorated with @pytest.fixture
# pytest creates a FRESH instance of each fixture for every test that requests it

@pytest.fixture
def sample_user():
    """Returns a fresh user dict for each test that requests it."""
    print('\n  [FIXTURE] Creating sample_user')   # only visible when running with: pytest -s
    return {'id': 1, 'name': 'Alice', 'email': 'alice@example.com', 'active': True}


@pytest.fixture
def sample_numbers():
    return [10, 20, 30, 40, 50]


# ─── Tests requesting a single fixture ───────────────────────────────────────
# pytest sees 'sample_user' in the parameter list, finds the fixture by that
# name, calls it, and injects the return value — no Depends(), no imports needed

def test_user_name(sample_user):
    assert sample_user['name'] == 'Alice'

def test_user_is_active(sample_user):
    assert sample_user['active'] is True      # 'is True' checks identity, not just truthiness

def test_user_email_has_at(sample_user):
    assert '@' in sample_user['email']         # substring check on email format

def test_numbers_count(sample_numbers):
    assert len(sample_numbers) == 5            # sample_user fixture never runs for this test

def test_numbers_sum(sample_numbers):
    assert sum(sample_numbers) == 150          # 10+20+30+40+50

# ─── Requesting multiple fixtures ────────────────────────────────────────────
# just add more parameters — pytest injects each one independently

def test_user_and_numbers(sample_user, sample_numbers):
    assert sample_user['id'] == 1
    assert sample_numbers[0] == 10


# ─── Fixtures depending on other fixtures ────────────────────────────────────
# fixtures can REQUEST other fixtures the same way tests do —
# just declare them as parameters, pytest resolves the whole chain automatically

@pytest.fixture
def user_with_scores(sample_user, sample_numbers):
    """Builds on sample_user and sample_numbers — pytest injects both."""
    sample_user['scores'] = sample_numbers     # attach numbers list to user dict
    return sample_user                         # return the enriched user


def test_user_scores(user_with_scores):
    assert 'scores' in user_with_scores                # key was added by fixture
    assert user_with_scores['scores'][2] == 30         # index 2 → 30 (third item)

Writing test_fixtures_basic.py


In [7]:
!pytest test_fixtures_basic.py -v -s

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 7 items                                                              

test_fixtures_basic.py::test_user_name 
  [FIXTURE] Creating sample_user
PASSED
test_fixtures_basic.py::test_user_is_active 
  [FIXTURE] Creating sample_user
PASSED
test_fixtures_basic.py::test_user_email_has_at 
  [FIXTURE] Creating sample_user
PASSED
test_fixtures_basic.py::test_numbers_count PASSED
test_fixtures_basic.py::test_numbers_sum PASSED
test_fixtures_basic.py::test_user_and_numbers 
  [FIXTURE] Creating sample_user
PASSED
test_fixtures_basic.py::test_user_scores 
  [FIXTURE] Creating sample_

## 7. Fixture Scope — How Often Is the Fixture Created?

By default, each fixture is **created fresh for every test** (scope=`function`). But some resources are expensive to create (DB connections, ML models). You can control this with `scope=`.

| Scope | Created once per | Destroyed after | Use for |
|-------|-----------------|------------------|---------|
| `function` | **Each test** (default) | Each test ends | Simple data, cheap setup |
| `class` | Each test class | Class ends | Shared setup in `TestXxx` classes |
| `module` | Each `.py` file | File ends | DB connections per file |
| `session` | Entire test run | All tests end | Expensive: ML models, auth tokens |

**Rule of thumb:** Start with `function` (default). Widen scope only if setup is slow.

**Important:** A wider-scope fixture can only use fixtures of the same or wider scope. A `session` fixture cannot use a `function` fixture.

In [8]:
%%writefile test_fixture_scope.py
import pytest


# ─── Fixture Scope ────────────────────────────────────────────────────────────
# scope controls HOW OFTEN a fixture is created and destroyed
#
# scope='function'  → new instance for EVERY test        (default)
# scope='module'    → one instance for the entire FILE
# scope='session'   → one instance for the entire pytest run (all files)


@pytest.fixture(scope='function')   # default — recreated before EACH test, destroyed after
def function_data():
    print('\n  [function] setup')
    data = {'count': 0}
    yield data                       # test runs here — receives fresh dict every time
    print('\n  [function] teardown') # runs after EACH test that uses this fixture


@pytest.fixture(scope='module')     # created ONCE when first test in this file needs it
def module_data():                  # destroyed after the last test in this file finishes
    print('\n  [module] setup — runs ONCE for entire file')
    data = {'shared': True, 'count': 0}
    yield data                       # same dict object shared across all tests in this file
    print('\n  [module] teardown')


@pytest.fixture(scope='session')    # created ONCE when pytest starts
def session_data():                 # destroyed only after ALL tests in ALL files finish
    print('\n  [session] setup — runs ONCE for entire pytest session')
    data = {'initialized': True}
    yield data                       # same dict object shared across every test in every file
    print('\n  [session] teardown')


# ─── function scope — each test gets its OWN fresh copy ──────────────────────

def test_function_scope_a(function_data):
    function_data['count'] += 1
    assert function_data['count'] == 1     # starts at 0 → incremented to 1

def test_function_scope_b(function_data):
    function_data['count'] += 1
    assert function_data['count'] == 1     # FRESH dict — count starts at 0 again
                                            # changes from test_a did NOT carry over


# ─── module scope — all tests in this file share the SAME object ──────────────

def test_module_scope_a(module_data):
    module_data['count'] += 1
    assert module_data['count'] == 1       # first test to use it — count goes 0 → 1

def test_module_scope_b(module_data):
    module_data['count'] += 1
    assert module_data['count'] == 2       # SAME dict — count carries over from test_a
                                            # this is the key difference from function scope


# ─── session scope — shared across ALL files in the entire test run ───────────

def test_session_scope(session_data):
    assert session_data['initialized'] is True   # value set during fixture setup, still there

Writing test_fixture_scope.py


In [9]:
# -s shows print() output so you can SEE when each fixture is created/destroyed
!pytest test_fixture_scope.py -v -s

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 5 items                                                              

test_fixture_scope.py::test_function_scope_a 
  [function] setup
PASSED
  [function] teardown

test_fixture_scope.py::test_function_scope_b 
  [function] setup
PASSED
  [function] teardown

test_fixture_scope.py::test_module_scope_a 
  [module] setup — runs ONCE for entire file
PASSED
test_fixture_scope.py::test_module_scope_b PASSED
test_fixture_scope.py::test_session_scope 
  [session] setup — runs ONCE for entire pytest session
PASSED
  [module] teardown

  [session] teardown


======================

## 8. `yield` Fixtures — Setup AND Teardown in One Function

A fixture with `yield` splits into setup (before yield) and teardown (after yield).
This replaces `setUp` + `tearDown` from unittest, but in a single readable function.

```python
@pytest.fixture
def database():
    # --- SETUP: runs before the test ---
    db = Database.connect('sqlite:///test.db')
    db.create_tables()

    yield db          # <-- the test runs here, receives the db object

    # --- TEARDOWN: runs after the test (even if test fails) ---
    db.rollback()
    db.close()
```

**Key guarantee:** The teardown ALWAYS runs, even if the test raises an exception. This is safer than `try/finally` patterns.

In [ ]:
%%writefile test_yield_fixture.py
import pytest


# ─── Simulated resource ───────────────────────────────────────────────────────
# Represents anything that needs explicit cleanup — DB connection, file handle,
# network socket, etc. The key idea: it must be CLOSED after use or it leaks.

class FakeDatabase:
    def __init__(self):
        self.data = []
        self.is_open = True
        print('\n  [DB] Connection opened')

    def insert(self, record):
        if not self.is_open:
            raise RuntimeError('DB is closed')   # guard — can't write to a closed DB
        self.data.append(record)

    def count(self):
        return len(self.data)                     # how many records currently in DB

    def close(self):
        self.is_open = False
        self.data.clear()                         # wipe data on close
        print('\n  [DB] Connection closed, data cleared')

# ─── yield fixture — setup + teardown in one function ────────────────────────
# everything BEFORE yield = setup   (runs before the test)
# everything AFTER  yield = teardown (runs after the test, even if test FAILS)
# this guarantees db.close() is always called — no connection leaks

@pytest.fixture
def db():
    database = FakeDatabase()   # ① open connection — setup
    yield database              # ② test runs here — receives the database object
    database.close()            # ③ close connection — teardown, guaranteed to run


# ─── Tests ────────────────────────────────────────────────────────────────────
# each test gets a BRAND NEW FakeDatabase (function scope — default)
# so data from one test never bleeds into another

def test_insert_one_record(db):
    db.insert({'id': 1, 'name': 'Alice'})
    assert db.count() == 1                 # inserted 1 record → count should be 1

def test_insert_multiple_records(db):
    # fresh db here — previous test's data is gone
    db.insert({'id': 1, 'name': 'Alice'})
    db.insert({'id': 2, 'name': 'Bob'})
    assert db.count() == 2                 # 2 inserts → count should be 2

def test_empty_database(db):
    assert db.count() == 0                 # fresh db, nothing inserted yet → 0

Writing test_yield_fixture.py


In [11]:
!pytest test_yield_fixture.py -v -s

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 3 items                                                              

test_yield_fixture.py::test_insert_one_record 
  [DB] Connection opened
PASSED
  [DB] Connection closed, data cleared

test_yield_fixture.py::test_insert_multiple_records 
  [DB] Connection opened
PASSED
  [DB] Connection closed, data cleared

test_yield_fixture.py::test_empty_database 
  [DB] Connection opened
PASSED
  [DB] Connection closed, data cleared


============================== 3 passed in 0.01s ===============================


## 9. `conftest.py` — The Shared Fixture Hub

### The Problem It Solves

Without `conftest.py`, if 10 test files all need a `db` fixture,
you'd have to define it in every single file — or import it manually.
`conftest.py` is pytest's way of saying: **"put shared fixtures here,
and every test file nearby gets them automatically."**

---

### How It Works

```
backend/
  conftest.py              ← fixtures here available to ALL test files below
  tests/
    conftest.py            ← fixtures here available only inside tests/
    test_users.py          ← can use fixtures from BOTH conftest.py files
    test_products.py       ← same
```

pytest automatically finds and loads every `conftest.py` it encounters
while collecting tests — no import needed in your test files.

```python
# conftest.py — define once
@pytest.fixture
def db():
    database = FakeDatabase()
    yield database
    database.close()

# test_users.py — just request it by name, no import
def test_create_user(db):       # pytest finds db in conftest.py automatically
    db.insert({'name': 'Alice'})
    assert db.count() == 1

# test_products.py — same fixture, no import here either
def test_create_product(db):
    db.insert({'name': 'Laptop'})
    assert db.count() == 1
```

---

### Hierarchical Scoping — Closer Wins

When two `conftest.py` files define a fixture with the same name,
the **closer one wins**:

```
backend/conftest.py     →  db fixture (real DB, used in production tests)
tests/conftest.py       →  db fixture (fake DB, overrides for unit tests)

test_users.py uses      →  tests/conftest.py version  (closer)
```

---

### What Goes Where

```
conftest.py         → shared fixtures used by many test files (db, client, auth tokens)
test_*.py           → fixtures only one test file needs
```

---

### Rules

```
✅ define shared fixtures in conftest.py
✅ multiple conftest.py files are fine — one per directory level
❌ never import from conftest.py — pytest loads it automatically
❌ don't put test functions in conftest.py — only fixtures and hooks
```

---

> 💡 Think of `conftest.py` as the `main.py` of your test suite —
> it doesn't run tests itself, it just wires up the shared resources
> that all other test files can plug into.

In [ ]:
%%writefile conftest.py
# conftest.py — shared fixtures for ALL test files in this directory
# pytest discovers and loads this file automatically — never import it manually

import pytest


@pytest.fixture
def base_user():
    """Available in ALL test files in this directory — no import needed."""
    return {'id': 99, 'name': 'Shared User', 'role': 'user'}


@pytest.fixture
def user_list():
    # a reusable list of users — any test file in this directory can request it
    return [
        {'id': 1, 'name': 'Alice', 'role': 'admin'},
        {'id': 2, 'name': 'Bob',   'role': 'user'},
        {'id': 3, 'name': 'Carol', 'role': 'user'},
    ]


@pytest.fixture
def admin_user(user_list):
    # fixtures can depend on other fixtures — pytest resolves the chain automatically
    # user_list is injected here just like it would be in a test function
    # next() just picks the first item from a loop that matches the condition
    return next(u for u in user_list if u['role'] == 'admin')  # it returns first user where role == 'admin'

Writing conftest.py


In [13]:
%%writefile test_uses_conftest.py
# Notice: no import of conftest.py anywhere in this file
# pytest scans the directory, finds conftest.py, and makes all its
# fixtures available here automatically — just request by name as usual

def test_base_user_name(base_user):           # 'base_user' defined in conftest.py
    assert base_user['name'] == 'Shared User'

def test_user_list_count(user_list):           # 'user_list' defined in conftest.py
    assert len(user_list) == 3                 # 3 users in the list

def test_admin_is_alice(admin_user):           # 'admin_user' depends on 'user_list'
    # pytest resolves the full chain: user_list → admin_user → this test
    assert admin_user['name'] == 'Alice'       # only Alice has role='admin'
    assert admin_user['role'] == 'admin'

def test_regular_users(user_list):
    regular = [u for u in user_list if u['role'] == 'user']   # filter out admins
    assert len(regular) == 2                   # Bob and Carol are regular users

Writing test_uses_conftest.py


In [14]:
!pytest test_uses_conftest.py -v

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 4 items                                                              

test_uses_conftest.py::test_base_user_name PASSED                        [ 25%]
test_uses_conftest.py::test_user_list_count PASSED                       [ 50%]
test_uses_conftest.py::test_admin_is_alice PASSED                        [ 75%]
test_uses_conftest.py::test_regular_users PASSED                         [100%]

============================== 4 passed in 0.01s ===============================


## 10. `@pytest.mark.parametrize` — Data-Driven Tests

### The Problem Without It

```python
# same logic, copy-pasted 3 times — only the inputs differ
def test_add_1_plus_1(): assert add(1, 1) == 2
def test_add_2_plus_3(): assert add(2, 3) == 5
def test_add_neg():      assert add(-1, 1) == 0
```

If `add()` changes, you update logic in 3 places. If you want 10 cases,
you write 10 functions. `parametrize` collapses all of this into one.

---

### With `parametrize` — One Function, Many Inputs

```python
@pytest.mark.parametrize('a, b, expected', [  # declare parameter names
    (1,  1,  2),                               # test case 1
    (2,  3,  5),                               # test case 2
    (-1, 1,  0),                               # test case 3
])
def test_add(a, b, expected):
    assert add(a, b) == expected               # pytest runs this 3 times, once per row
```

pytest treats each tuple as a **separate test** — they appear
individually in the output, fail independently, and are reported with
their own input values.

---

### How to Read the Decorator

```python
@pytest.mark.parametrize('a, b, expected', [...])
#                         ^^^^^^^^^^^^^^    ^^^
#                         parameter names   list of tuples
#                         (comma-separated  (one tuple = one test run)
#                          as a string)
```

Name count must match tuple length — `'a, b, expected'` = 3 names,
so every tuple must have exactly 3 values.

---

### Single Parameter — No Tuple Needed

```python
@pytest.mark.parametrize('email', [
    'alice@example.com',
    'bob@test.org',
    'invalid-email',       # should fail validation
])
def test_email_format(email):
    ...                    # one string per run, no tuple needed for single param
```

---

### Why This Is Huge for ML/Data Work

```python
# test your preprocessing function on many input types at once
@pytest.mark.parametrize('raw, expected', [
    ('  Alice  ', 'alice'),        # strips whitespace, lowercases
    ('BOB',       'bob'),          # handles all caps
    ('',          ''),             # handles empty string
    (None,        ''),             # handles None
])
def test_clean_name(raw, expected):
    assert clean_name(raw) == expected
```

One function covers edge cases, normal cases, and failure cases —
all visible at a glance as a data table.

---

> 💡 `parametrize` = same test logic, different data. If you find
> yourself copy-pasting a test and only changing the inputs, that's
> the signal to use `parametrize` instead.

In [1]:
%%writefile test_parametrize.py
import pytest


def normalize_text(text):
    """Lowercase, strip whitespace, remove punctuation."""
    import re
    return re.sub(r'[^a-z0-9 ]', '', text.lower().strip())

def is_valid_email(email):
    import re
    return bool(re.match(r'^[\w.+-]+@[\w-]+\.[a-z]{2,}$', email))

def clamp(value, min_val, max_val):
    """Clamp a value within [min_val, max_val]."""
    return max(min_val, min(max_val, value))


# Single parameter
@pytest.mark.parametrize('text', [
    'Hello World',
    'HELLO WORLD',
    '  hello world  ',
])
def test_normalize_produces_lowercase(text):
    result = normalize_text(text)
    assert result == result.lower()


# Multiple parameters: (input, expected_output)
@pytest.mark.parametrize('text, expected', [
    ('Hello, World!', 'hello world'),
    ('  PYTHON  ',    'python'),
    ('ML-Model v2!',  'mlmodel v2'),
    ('',              ''),
])
def test_normalize_text(text, expected):
    assert normalize_text(text) == expected


# Valid and invalid emails — same function, both cases
@pytest.mark.parametrize('email, expected', [
    ('user@example.com',     True),
    ('alice@company.org',    True),
    ('name+tag@domain.io',   True),
    ('notanemail',           False),
    ('@nodomain.com',        False),
    ('missing@.com',         False),
])
def test_email_validation(email, expected):
    assert is_valid_email(email) == expected


# Three parameters
@pytest.mark.parametrize('value, lo, hi, expected', [
    (5, 0, 10,  5),     # within range
    (-3, 0, 10, 0),     # below min, clamped to 0
    (99, 0, 10, 10),    # above max, clamped to 10
    (0, 0, 10,  0),     # exactly at min
    (10, 0, 10, 10),    # exactly at max
])
def test_clamp(value, lo, hi, expected):
    assert clamp(value, lo, hi) == expected

Writing test_parametrize.py


In [2]:
!pytest test_parametrize.py -v

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 18 items                                                             

test_parametrize.py::test_normalize_produces_lowercase[Hello World] PASSED [  5%]
test_parametrize.py::test_normalize_produces_lowercase[HELLO WORLD] PASSED [ 11%]
test_parametrize.py::test_normalize_produces_lowercase[  hello world  ] PASSED [ 16%]
test_parametrize.py::test_normalize_text[Hello, World!-hello world] PASSED [ 22%]
test_parametrize.py::test_normalize_text[  PYTHON  -python] PASSED       [ 27%]
test_parametrize.py::test_normalize_text[ML-Model v2!-mlmodel v2] PASSED [ 33%]
test_parametrize

## 11. Markers — Categorize and Control Tests

Markers are labels you put on tests to control **which ones run** and
**how pytest treats them**. Two kinds: built-in markers that change
behavior, and custom markers that just tag tests for filtering.

---

### Built-in Markers

```python
import sys

# skip — always skip, never runs
@pytest.mark.skip(reason='feature not implemented yet')
def test_export_pdf():
    ...

# skipif — skip only when a condition is true
@pytest.mark.skipif(sys.platform == 'win32', reason='Linux only')
def test_file_permissions():
    ...

# xfail — expected to fail
# marks as 'xfail' (not an error) if it fails, 'xpass' if it surprisingly passes
@pytest.mark.xfail(reason='known bug in payment gateway, ticket #123')
def test_payment_processing():
    ...
```

```
PASSED   → test passed normally
FAILED   → test failed unexpectedly
SKIPPED  → test was skipped
xfail    → test failed, but that was expected  ✅
xpass    → test passed, but we expected it to fail  ⚠️
```

---

### Custom Markers — Tag Tests for Filtering

```python
@pytest.mark.slow
def test_train_model():
    ...                    # takes 30 seconds — don't want this in every run

@pytest.mark.integration
def test_payment_api():
    ...                    # hits a real external service
```

Run only what you need:

```bash
pytest -m slow              # run ONLY slow tests
pytest -m "not slow"        # skip slow tests, run everything else
pytest -m "integration"     # run only integration tests
pytest -m "not integration" # skip integration tests
```

---

### Register Custom Markers in `pytest.ini`

Without registration pytest still works, but shows a warning for every
unknown marker. Registration documents what each marker means:

```ini
[pytest]
markers =
    slow: marks tests as slow (deselect with '-m "not slow"')
    integration: marks tests that require external services
```

---

### When to Use Which

```
skip      → test is broken or feature doesn't exist yet
skipif    → test only makes sense on certain OS / Python version
xfail     → known bug exists — test documents it without blocking CI
custom    → organize tests into groups you can run selectively
```

---

> 💡 `xfail` is especially useful in teams — instead of deleting a
> failing test or commenting it out, mark it `xfail` with a ticket
> number. The test stays in the suite and documents the known bug.

In [3]:
%%writefile pytest_markers.ini
[pytest]
markers =
    slow: marks tests as slow (run with: pytest -m slow)
    fast: marks tests as fast unit tests
    integration: marks tests that require external services
    # registering markers here silences "unknown marker" warnings
    # also serves as documentation — anyone reading this knows what markers exist

Writing pytest_markers.ini


In [4]:
%%writefile test_markers.py
import sys
import pytest


# ─── @pytest.mark.skip — unconditional skip ───────────────────────────────────
# test body never executes — shows as SKIPPED in output, not FAILED
# use when a feature doesn't exist yet or test is temporarily broken

@pytest.mark.skip(reason='feature not implemented yet')
def test_future_feature():
    assert False   # never runs — skip happens before the function is called


# ─── @pytest.mark.skipif — conditional skip ──────────────────────────────────
# evaluates the condition at collection time — skips only when condition is True
# useful for OS-specific, Python-version-specific, or environment-specific tests

@pytest.mark.skipif(sys.platform == 'win32', reason='Linux only')
def test_linux_feature():
    assert True    # runs on Linux/Mac, skipped on Windows


# ─── @pytest.mark.xfail — expected failure ───────────────────────────────────
# use when a known bug exists but hasn't been fixed yet
# test failing = expected → marked xfail  ✅ (not an error)
# test passing = surprise → marked xpass  ⚠️ (may need attention)

@pytest.mark.xfail(reason='known bug: division returns wrong type')
def test_known_bug():
    result = 10 / 3
    assert isinstance(result, int)   # fails — 10/3 is float, not int
                                      # but marked xfail so CI stays green


# ─── @pytest.mark.xfail with strict=True ─────────────────────────────────────
# strict=False (default) → xpass is a warning, test suite still passes
# strict=True            → xpass is an ERROR — forces you to remove the xfail marker
#                          once the bug is fixed, keeping markers honest

@pytest.mark.xfail(strict=True, reason='bug #42 is still open')
def test_strict_xfail():
    assert 1 + 1 == 3              # fails → xfail as expected → overall: pass
                                    # if this ever passes → strict=True makes it an ERROR


# ─── Custom markers — tag tests for selective running ─────────────────────────
# these don't change behavior on their own — they're labels for filtering
# run with: pytest -m slow / pytest -m "not slow" / pytest -m integration

@pytest.mark.slow
def test_heavy_computation():
    # simulates a slow test — excluded from quick runs via: pytest -m "not slow"
    total = sum(range(1_000_000))
    assert total == 499999500000


@pytest.mark.fast
def test_simple_addition():
    # pure unit test — no I/O, no setup, runs in milliseconds
    assert 1 + 1 == 2


@pytest.mark.integration
def test_requires_external_service():
    # would hit a real API or DB in production
    # skipped here because no external service is available in this demo
    pytest.skip('no external service in this demo')

Writing test_markers.py


In [ ]:
# run pytest on test_markers.py with these flags:
#
# -v                → verbose — shows each test name and its result individually
# -rs               → show SHORT reason for SKIPPED tests in the summary
# -p no:cacheprovider → disable the cache plugin — no .pytest_cache folder written
#                       useful in notebooks where you don't want cache files cluttering up

!pytest test_markers.py -v -rs -p no:cacheprovider

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 7 items                                                              

test_markers.py::test_future_feature SKIPPED (feature not implemente...) [ 14%]
test_markers.py::test_linux_feature PASSED                               [ 28%]
test_markers.py::test_known_bug XFAIL (known bug: division returns w...) [ 42%]
test_markers.py::test_strict_xfail XFAIL (bug #42 is still open)         [ 57%]
test_markers.py::test_heavy_computation PASSED                           [ 71%]
test_markers.py::test_simple_addition PASSED                             [ 85%]
test_markers.py::test_requires_external_service SKIP

In [6]:
# Run only 'slow' tests
!pytest test_markers.py -v -m slow -p no:cacheprovider

# Run everything EXCEPT slow tests (common in CI for quick feedback):
# !pytest -m 'not slow'

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 7 items / 6 deselected / 1 selected                                  

test_markers.py::test_heavy_computation PASSED                           [100%]

=============================== warnings summary ===============================
test_markers.py:50
  /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks/test_markers.py:50: PytestUnknownMarkWarning: Unknown pytest.mark.slow - is this a typo?  You can register custom marks to avoid this warning - for details, see https://docs.pytest.org/en/stable/how-to/mark.html
    @pytest.mark.slow

test_m

## 12. Mocking — Why Mock?

### The Problem

Unit tests should test **one thing in isolation**. But functions often
call out to things you can't control:

```python
def get_weather(city):
    response = requests.get(f"https://api.weather.com/{city}")  # real network call
    return response.json()['temp']

def test_get_weather():
    assert get_weather("London") == 15    # ❌ fails if internet is down
                                           # ❌ slow, costs API credits
                                           # ❌ result changes every run
```

You don't want to test the weather API — you want to test **your
function's logic**. Mocking replaces the real dependency with a
controlled fake.

```
Real flow:    test → function → requests.get → internet → unpredictable response
Mocked flow:  test → function → Mock()       → returns exactly what you told it to
```

---

### When to Mock vs When Not To

```
✅ Mock:       external HTTP calls, email sending, payment APIs,
               file I/O, datetime.now(), random numbers
❌ Don't mock: the function you're actually testing,
               simple data transformations, standard library math
⚠️  Integration tests: DON'T mock — use a real test DB instead
```

---

## 13. `Mock` vs `MagicMock`

Both come from `unittest.mock` — built into Python, no install needed.

```python
from unittest.mock import Mock, MagicMock
```

| | `Mock` | `MagicMock` |
|-|--------|-------------|
| Basic attribute access | ✅ | ✅ |
| Method calls | ✅ | ✅ |
| Dunder methods (`__len__`, `__iter__`, `__enter__`) | ❌ | ✅ |
| Context managers (`with` statement) | ❌ | ✅ |
| Iterables (`for x in mock`) | ❌ | ✅ |

---

### `Mock` — Basic Fake Object

```python
m = Mock()
m.some_method.return_value = 42    # tell it what to return
result = m.some_method()
assert result == 42                # returns exactly what you told it

m.some_method("hello")             # records every call
m.some_method.assert_called_once_with("hello")   # verify it was called
```

---

### `MagicMock` — When You Need Dunder Methods

```python
# Mock breaks with context managers:
m = Mock()
with m:    # ❌ TypeError — Mock doesn't support __enter__ / __exit__
    pass

# MagicMock handles it:
m = MagicMock()
with m:    # ✅ works — MagicMock has __enter__ / __exit__ pre-configured
    pass
```

---

### `spec=` — Prevent Typo Bugs

Without `spec`, mocks accept ANY attribute — even ones that don't exist
on the real object, silently masking typos:

```python
m = Mock()
m.send_emial()       # typo — but Mock accepts it silently ❌

m = Mock(spec=EmailService)
m.send_emial()       # ✅ AttributeError — 'send_emial' doesn't exist on EmailService
                      # catches the typo immediately
```

---

### Rule of Thumb

```
Replacing a context manager or iterable?  → MagicMock
Everything else?                           → Mock
Want typo protection?                      → add spec=RealClass to either
```

> 💡 `MagicMock` is a subclass of `Mock` — it does everything `Mock`
> does, plus dunder method support. When in doubt, `MagicMock` is the
> safer default.

In [7]:
from unittest.mock import Mock, MagicMock

# ─── Mock basics ──────────────────────────────────────────────────────────────
m = Mock()

# accessing ANY attribute on a Mock returns another Mock — never raises AttributeError
# this is what makes mocks so flexible — they accept anything by default
print(type(m.anything))          # <class 'unittest.mock.Mock'>

# calling ANY method also returns a Mock — no setup needed to avoid crashes
result = m.send_email('alice@example.com', 'Hello')
print(type(result))              # <class 'unittest.mock.Mock'>

# but Mock silently records every call — you can inspect them later
print(m.send_email.call_count)   # 1 — called once
print(m.send_email.call_args)    # call('alice@example.com', 'Hello') — exact args used

print('---')

# ─── spec= prevents typos ─────────────────────────────────────────────────────
# without spec, Mock accepts any attribute name — typos go unnoticed
# with spec=RealClass, accessing attributes NOT on the real class raises AttributeError

class EmailService:
    def send(self, to, subject, body): pass
    def is_connected(self): pass

strict_mock = Mock(spec=EmailService)
strict_mock.send('a@b.com', 'hi', 'body')   # ✅ 'send' exists on EmailService — allowed

try:
    strict_mock.typo_method()               # ❌ 'typo_method' not on EmailService
except AttributeError as e:
    print(f'spec caught typo: {e}')          # spec acts as a safety net for typos

<class 'unittest.mock.Mock'>
<class 'unittest.mock.Mock'>
1
call('alice@example.com', 'Hello')
---
spec caught typo: Mock object has no attribute 'typo_method'


In [8]:
from unittest.mock import Mock, MagicMock

# ─── return_value — control what a mock returns ───────────────────────────────
# instead of hitting a real DB, tell the mock exactly what to hand back
mock_db = Mock()
mock_db.get_user.return_value = {'id': 1, 'name': 'Alice'}   # every call returns this

user = mock_db.get_user(1)
print(user)              # {'id': 1, 'name': 'Alice'} — no DB involved

print('---')

# ─── side_effect: sequence of values ─────────────────────────────────────────
# return_value always returns the SAME thing
# side_effect with a list returns values ONE BY ONE — useful for simulating
# paginated responses, retries, or changing state over multiple calls

mock_api = Mock()
mock_api.fetch.side_effect = [100, 200, 300]   # first call→100, second→200, third→300

print(mock_api.fetch())  # 100
print(mock_api.fetch())  # 200
print(mock_api.fetch())  # 300

print('---')

# ─── side_effect: raise an exception ─────────────────────────────────────────
# use this to simulate failures — network errors, timeouts, DB failures
# lets you test that your error handling code actually works

mock_flaky = Mock()
mock_flaky.call.side_effect = ConnectionError('Network unavailable')

try:
    mock_flaky.call()                        # raises the exception you set
except ConnectionError as e:
    print(f'Got expected error: {e}')

print('---')

# ─── side_effect: callable ────────────────────────────────────────────────────
# when side_effect is a FUNCTION, it runs on every call with the same args
# useful when you need dynamic behavior — inspect, transform, or track inputs

calls = []
def capture_call(arg):
    calls.append(arg)               # side effect: record what was passed in
    return f'processed:{arg}'       # still returns a controlled value

mock_proc = Mock()
mock_proc.process.side_effect = capture_call

print(mock_proc.process('hello'))   # processed:hello — return value from capture_call
print(calls)                         # ['hello'] — captured what was passed in

{'id': 1, 'name': 'Alice'}
---
100
200
300
---
Got expected error: Network unavailable
---
processed:hello
['hello']


In [9]:
from unittest.mock import Mock

# ─── Mock assertion methods — verify HOW a mock was called ────────────────────
# this is the other half of mocking — not just controlling output,
# but VERIFYING that your function called its dependencies correctly

mock_service = Mock()
mock_service.notify('alice@example.com', message='Welcome!')

# was the method called at least once? (doesn't care about args)
mock_service.notify.assert_called()
print('assert_called: passed')

# was it called EXACTLY once? (still doesn't check args)
mock_service.notify.assert_called_once()
print('assert_called_once: passed')

# was it called exactly once WITH these specific arguments?
# most precise — checks both call count AND arguments
mock_service.notify.assert_called_once_with('alice@example.com', message='Welcome!')
print('assert_called_once_with: passed')

# manual call count check — useful when you expect N calls, not just 1
assert mock_service.notify.call_count == 1

# verify a method was NEVER called — useful for negative testing
# e.g. confirm delete wasn't triggered when it shouldn't have been
mock_service.delete.assert_not_called()
print('assert_not_called: passed')

# ─── inspect multiple calls ───────────────────────────────────────────────────
# when a method is called multiple times, call_args only shows the LAST call
# call_args_list shows ALL calls in order — useful for verifying sequences

mock_service.send('msg1')
mock_service.send('msg2')
mock_service.send('msg3')
print('all send calls:', mock_service.send.call_args_list)
# [call('msg1'), call('msg2'), call('msg3')]

assert_called: passed
assert_called_once: passed
assert_called_once_with: passed
assert_not_called: passed
all send calls: [call('msg1'), call('msg2'), call('msg3')]


## 14. `patch()` — Replacing Real Objects in Tests

`Mock()` creates a fake object you control.
`patch()` **swaps out a real object with a mock** for the duration of
a test — then restores the original automatically.

---

### Two Ways to Use `patch()`

```python
from unittest.mock import patch

# Form 1 — decorator (cleaner for single patches)
@patch('mymodule.requests.get')       # replace requests.get inside mymodule
def test_something(mock_get):         # mock is injected as the last parameter
    mock_get.return_value.status_code = 200
    ...
# real requests.get is restored after the test finishes


# Form 2 — context manager (cleaner when you need the mock mid-test)
def test_something():
    with patch('mymodule.requests.get') as mock_get:
        mock_get.return_value.status_code = 200
        ...
    # real requests.get is restored here, after the with block
```

Both forms do the same thing — just different syntax for different
situations. The mock only exists for the duration of the test.

---

## 15. The Golden Rule of Patching

> **Patch where the name is USED, not where it is DEFINED.**

This is the #1 source of confusion with mocking. Here's why it matters:

```python
# user_service.py
import requests                    # requests is imported INTO user_service's namespace

def get_user(id):
    return requests.get(f'/users/{id}')   # 'requests' here is user_service's own reference
```

```python
# ❌ WRONG — patching at the source
with patch('requests.get') as m:
    ...
# This replaces requests.get in the requests package itself
# but user_service already has its OWN reference to the original
# so user_service.requests.get still points to the real function — patch does nothing


# ✅ CORRECT — patching where it's used
with patch('user_service.requests.get') as m:
    ...
# This replaces requests.get inside user_service's namespace
# now when get_user() calls requests.get, it hits your mock
```

---

### Mental Model — Think of Namespaces as Rooms

```
requests package          user_service module
┌─────────────────┐       ┌──────────────────────┐
│ def get(url):   │       │ requests → [ points ] │──→ requests.get()
│   ...real code  │       │                      │
└─────────────────┘       └──────────────────────┘

patch('requests.get')           → changes the original room — user_service
                                  still points to the old reference ❌

patch('user_service.requests.get') → changes the pointer INSIDE user_service's
                                     room — now it points to your mock ✅
```

---

### Quick Rule

```
Where is the import?       →  that's where you patch

# user_service.py has: import requests
→ patch('user_service.requests.get')

# user_service.py has: from requests import get
→ patch('user_service.get')          # 'get' is now directly in user_service
```

---

> 💡 If your patch isn't working — your mock isn't being called —
> this rule is almost always the reason. Check where the name
> lives at the time the function runs, not where it was originally defined.

In [10]:
%%writefile user_service.py
# user_service.py — a module that depends on external requests
import requests


def get_user(user_id):
    """Fetch a user from an external API."""
    response = requests.get(f'https://api.example.com/users/{user_id}')
    if response.status_code != 200:
        raise ValueError(f'User {user_id} not found (status {response.status_code})')
    return response.json()


def is_premium_user(user_id):
    """Returns True if the user has premium status."""
    user = get_user(user_id)
    return user.get('premium', False)


def get_user_display_name(user_id):
    """Returns formatted display name."""
    user = get_user(user_id)
    first = user.get('first_name', '')
    last = user.get('last_name', '')
    return f'{first} {last}'.strip()

Writing user_service.py


In [11]:
%%writefile test_user_service.py
import pytest
from unittest.mock import patch, MagicMock
import user_service


# --- Test 1: Successful user fetch ---
def test_get_user_success():
    mock_response = MagicMock()
    mock_response.status_code = 200
    mock_response.json.return_value = {
        'id': 1, 'first_name': 'Alice', 'last_name': 'Smith', 'premium': True
    }

    # CORRECT: patch where requests is USED (in user_service module)
    with patch('user_service.requests.get', return_value=mock_response):
        user = user_service.get_user(1)

    assert user['first_name'] == 'Alice'
    assert user['premium'] is True


# --- Test 2: User not found (404) ---
def test_get_user_not_found():
    mock_response = MagicMock()
    mock_response.status_code = 404

    with patch('user_service.requests.get', return_value=mock_response):
        with pytest.raises(ValueError, match='User 99 not found'):
            user_service.get_user(99)


# --- Test 3: Decorator form of patch ---
@patch('user_service.requests.get')
def test_get_user_decorator_style(mock_get):   # mock injected as parameter
    mock_get.return_value.status_code = 200
    mock_get.return_value.json.return_value = {'id': 2, 'premium': False}

    user = user_service.get_user(2)
    assert user['id'] == 2

    # Verify the mock was called with the right URL
    mock_get.assert_called_once_with('https://api.example.com/users/2')


# --- Test 4: Mock at a higher level ---
# Instead of mocking requests.get, mock get_user itself (the dependency of is_premium_user)
def test_is_premium_user_true():
    with patch('user_service.get_user', return_value={'id': 1, 'premium': True}):
        result = user_service.is_premium_user(1)
    assert result is True

def test_is_premium_user_false():
    with patch('user_service.get_user', return_value={'id': 2, 'premium': False}):
        result = user_service.is_premium_user(2)
    assert result is False


# --- Test 5: Display name formatting ---
def test_display_name():
    with patch('user_service.get_user', return_value={'first_name': 'John', 'last_name': 'Doe'}):
        name = user_service.get_user_display_name(1)
    assert name == 'John Doe'

def test_display_name_only_first_name():
    with patch('user_service.get_user', return_value={'first_name': 'Alice', 'last_name': ''}):
        name = user_service.get_user_display_name(1)
    assert name == 'Alice'

Writing test_user_service.py


In [12]:
!pytest test_user_service.py -v

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 7 items                                                              

test_user_service.py::test_get_user_success PASSED                       [ 14%]
test_user_service.py::test_get_user_not_found PASSED                     [ 28%]
test_user_service.py::test_get_user_decorator_style PASSED               [ 42%]
test_user_service.py::test_is_premium_user_true PASSED                   [ 57%]
test_user_service.py::test_is_premium_user_false PASSED                  [ 71%]
test_user_service.py::test_display_name PASSED                           [ 85%]
test_user_service.py::test_d

## 16. Mocking `datetime.now()` and Other Tricky Builtins

Some things are hard to patch because they're so fundamental. `datetime.now()` is a classic example — it changes every millisecond, making tests non-deterministic.

The trick: **patch it in the module that uses it**, not in the `datetime` module itself.

In [13]:
%%writefile time_utils.py
from datetime import datetime


def get_greeting():
    hour = datetime.now().hour
    if hour < 12:
        return 'Good morning'
    elif hour < 18:
        return 'Good afternoon'
    else:
        return 'Good evening'


def get_timestamp_label():
    return datetime.now().strftime('%Y-%m-%d %H:%M')

Writing time_utils.py


In [14]:
%%writefile test_time_utils.py
from unittest.mock import patch
from datetime import datetime
import time_utils


def test_morning_greeting():
    # freeze time at 9:00 AM
    with patch('time_utils.datetime') as mock_dt:
        mock_dt.now.return_value = datetime(2025, 1, 1, 9, 0, 0)
        assert time_utils.get_greeting() == 'Good morning'


def test_afternoon_greeting():
    with patch('time_utils.datetime') as mock_dt:
        mock_dt.now.return_value = datetime(2025, 1, 1, 14, 0, 0)
        assert time_utils.get_greeting() == 'Good afternoon'


def test_evening_greeting():
    with patch('time_utils.datetime') as mock_dt:
        mock_dt.now.return_value = datetime(2025, 1, 1, 20, 0, 0)
        assert time_utils.get_greeting() == 'Good evening'

Writing test_time_utils.py


In [15]:
!pytest test_time_utils.py -v

============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.1.1, pluggy-1.6.0 -- /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/myenv/bin/python3.11
cachedir: .pytest_cache
rootdir: /Users/anishupreti2002icloud.com/Desktop/Learning/relearning-concepts/backend-and-systems/testing/notebooks
plugins: cov-7.1.0, anyio-4.13.0
collected 3 items                                                              

test_time_utils.py::test_morning_greeting PASSED                         [ 33%]
test_time_utils.py::test_afternoon_greeting PASSED                       [ 66%]
test_time_utils.py::test_evening_greeting PASSED                         [100%]

============================== 3 passed in 0.01s ===============================


---
## Summary — Cheat Sheet

### pytest Basics
```bash
pytest -v                  # verbose
pytest -s                  # show print output
pytest -k 'login'          # run tests matching 'login'
pytest -x                  # stop at first failure
pytest -m slow             # run only 'slow' marked tests
pytest -m 'not slow'       # skip slow tests
pytest --tb=short          # shorter tracebacks
```

### Fixtures
```python
@pytest.fixture                    # function scope (default)
@pytest.fixture(scope='module')    # module scope
@pytest.fixture(scope='session')   # session scope

# yield fixtures: setup / teardown
@pytest.fixture
def resource():
    r = open_resource()  # setup
    yield r              # test runs
    r.close()            # teardown (always runs)
```

### Parametrize
```python
@pytest.mark.parametrize('input, expected', [
    ('hello', 5),
    ('pytest', 6),
])
def test_length(input, expected):
    assert len(input) == expected
```

### Mocking
```python
from unittest.mock import Mock, MagicMock, patch

m = Mock()
m.method.return_value = 42
m.method.side_effect = [1, 2, 3]         # sequence
m.method.side_effect = ValueError('err') # always raises

m.method.assert_called_once_with(arg1, kwarg=val)
m.method.assert_not_called()

# Patch where it's USED, not where it's defined
with patch('mymodule.requests.get') as mock_get:
    mock_get.return_value.json.return_value = {'id': 1}

@patch('mymodule.requests.get')
def test_fn(mock_get):      # mock injected as parameter
    ...
```

### Key Rules
1. `function` scope = new fixture per test (safest default)
2. `conftest.py` = auto-shared fixtures, never import it
3. Patch **where it's used**: `patch('mymodule.requests.get')` not `patch('requests.get')`
4. `MagicMock` when you need context manager (`with`) or dunder methods
5. `spec=RealClass` to prevent mocks masking typos